# Validate TinyLlama HelpSteer2 Geometry Proxy

This Colab notebook runs the proxy-validity experiment for the TinyLlama HelpSteer2 LoRA adapters.

Goal: test whether geometry scores from `R_cos` and `R_gram` rank merge coefficients similarly to true preference-weighted ArmoRM utility `U_p`.

The workflow is deliberately split:

1. load the relationship matrices from notebook 05,
2. build and freeze a finite coefficient search set `B`,
3. run a geometry-only dynamic-range pre-check with no ArmoRM,
4. optionally collect the expensive ArmoRM reward matrix with caching,
5. run Spearman correlation between geometry scores and `U_p`.

ArmoRM is evaluation-only here. Its scores must not feed back into adapter training, data selection, relationship-matrix construction, or checkpoint choice.

## 1. Clone or update the repository

Run this first in Colab. On a local checkout, it keeps the current folder.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/NZhang137/master-thesis.git"
PROJECT_DIR = Path("/content/master-thesis")

if Path("/content").exists():
    if (PROJECT_DIR / ".git").is_dir():
        print(f"Updating repository at {PROJECT_DIR}")
        subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
    else:
        print(f"Cloning repository to {PROJECT_DIR}")
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    os.chdir(PROJECT_DIR)
else:
    print("Not running in Colab; using the current local checkout.")

print(f"Working directory: {Path.cwd()}")

## 2. Check runtime

The geometry-only part is CPU-safe. Reward collection loads TinyLlama and ArmoRM and should use a GPU runtime.

In [ ]:
import platform
import shutil
import subprocess

print(f"Python runtime: {platform.python_version()}")
print(f"Platform: {platform.platform()}")

if shutil.which("nvidia-smi"):
    print("GPU detected:")
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("No GPU detected. Geometry-only checks are fine; reward collection will be slow or impractical.")

## 3. Install dependencies

The versions follow the TinyLlama/ArmoRM Colab notebooks. ArmoRM relies on `transformers==4.40.0`.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "pandas==2.2.2",
        "numpy<2.1",
        "scipy",
        "protobuf>=5.29.1,<6.0.0",
        "transformers==4.40.0",
        "peft==0.10.0",
        "accelerate==0.29.3",
        "safetensors",
        "datasets",
        "pyyaml",
        "matplotlib",
    ],
    check=True,
)
print("Installed proxy-validation dependencies.")

## 4. Settings

`RUN_REWARD_COLLECTION = True` runs the full expensive ArmoRM reward collection. Set it to `False` only if you want the cheap geometry-only pre-check first.

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import sys
import time
import zipfile
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CONFIG_PATH = Path("configs/tinyllama_helpsteer2_armorm.yaml")
COSINE_MATRIX_CSV = Path("results/tinyllama_helpsteer2_R/tinyllama_helpsteer2_R_cos.csv")
GRAM_MATRIX_CSV = Path("results/tinyllama_helpsteer2_R/tinyllama_helpsteer2_R_gram.csv")

RESULTS_DIR = Path("results/tinyllama_helpsteer2_proxy_validation")
PLOTS_DIR = RESULTS_DIR / "plots"
SEARCH_SET_NPY = RESULTS_DIR / "search_set_B.npy"
GEOMETRY_PRECHECK_JSON = RESULTS_DIR / "geometry_precheck.json"
PREREGISTRATION_JSON = RESULTS_DIR / "proxy_validation_preregistration.json"
REWARD_CACHE_JSONL = RESULTS_DIR / "reward_matrix_cache.jsonl"
REWARD_MATRIX_NPY = RESULTS_DIR / "reward_matrix.npy"
PROXY_RESULTS_JSON = RESULTS_DIR / "proxy_validation_results.json"
PROXY_RESULTS_CSV = RESULTS_DIR / "proxy_validation_spearman_summary.csv"
SPEARMAN_PLOT_PNG = PLOTS_DIR / "proxy_validation_spearman_summary.png"
OUTPUT_ZIP = Path("tinyllama_helpsteer2_proxy_validation_outputs.zip")

# Upload result/adapters zips when running in a fresh Colab session.
RUN_INPUT_ZIP_UPLOAD = True
INPUT_ZIP_EXTRACT_ROOT = Path(".")

# Geometry search set B.
SEARCH_SET_SEED = 137
N_DIRICHLET = 64
DIRICHLET_ALPHA = 1.0

# Reward collection. True runs the full expensive ArmoRM evaluation.
RUN_REWARD_COLLECTION = True
NUM_REWARD_PROMPTS = 80
MAX_NEW_TOKENS = 256
REPETITION_PENALTY = 1.15
NO_REPEAT_NGRAM_SIZE = 5
HELDOUT_PROMPT_SPLIT = "validation"
PROXY_PROMPT_PATH = RESULTS_DIR / "proxy_validation_fixed_prompts.jsonl"

# Interpretation thresholds to write before collecting rewards.
PROXY_VALIDATED_RHO = 0.60
PROXY_WEAK_RHO = 0.30


def find_project_root(start: Path | None = None) -> tuple[Path, bool]:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir():
            return candidate, True
    return start, False


PROJECT_ROOT, REPO_ROOT_FOUND = find_project_root()
if PROJECT_ROOT != Path.cwd().resolve():
    os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = (PROJECT_ROOT / CONFIG_PATH).resolve()
COSINE_MATRIX_CSV = (PROJECT_ROOT / COSINE_MATRIX_CSV).resolve()
GRAM_MATRIX_CSV = (PROJECT_ROOT / GRAM_MATRIX_CSV).resolve()
RESULTS_DIR = (PROJECT_ROOT / RESULTS_DIR).resolve()
PLOTS_DIR = (PROJECT_ROOT / PLOTS_DIR).resolve()
SEARCH_SET_NPY = (PROJECT_ROOT / SEARCH_SET_NPY).resolve()
GEOMETRY_PRECHECK_JSON = (PROJECT_ROOT / GEOMETRY_PRECHECK_JSON).resolve()
PREREGISTRATION_JSON = (PROJECT_ROOT / PREREGISTRATION_JSON).resolve()
REWARD_CACHE_JSONL = (PROJECT_ROOT / REWARD_CACHE_JSONL).resolve()
REWARD_MATRIX_NPY = (PROJECT_ROOT / REWARD_MATRIX_NPY).resolve()
PROXY_RESULTS_JSON = (PROJECT_ROOT / PROXY_RESULTS_JSON).resolve()
PROXY_RESULTS_CSV = (PROJECT_ROOT / PROXY_RESULTS_CSV).resolve()
SPEARMAN_PLOT_PNG = (PROJECT_ROOT / SPEARMAN_PLOT_PNG).resolve()
OUTPUT_ZIP = (PROJECT_ROOT / OUTPUT_ZIP).resolve()
PROXY_PROMPT_PATH = (PROJECT_ROOT / PROXY_PROMPT_PATH).resolve()
INPUT_ZIP_EXTRACT_ROOT = (PROJECT_ROOT / INPUT_ZIP_EXTRACT_ROOT).resolve()

print(f"Project root: {PROJECT_ROOT}")
print(f"Run reward collection: {RUN_REWARD_COLLECTION}")
print(f"Cosine matrix CSV: {COSINE_MATRIX_CSV}")
print(f"Gram matrix CSV: {GRAM_MATRIX_CSV}")
print(f"Results dir: {RESULTS_DIR}")

In [ ]:
# v2 metric and confirmatory-output paths.
_v2_results_rel = RESULTS_DIR.relative_to(PROJECT_ROOT)
METRICS_V2_JSON = (PROJECT_ROOT / _v2_results_rel / "proxy_validation_metrics_v2.json").resolve()
PREREGISTRATION_V2_JSON = (PROJECT_ROOT / _v2_results_rel / "proxy_validation_preregistration_v2.json").resolve()
CONFIRM_PROMPT_PATH = (PROJECT_ROOT / _v2_results_rel / "confirmatory_fixed_prompts.jsonl").resolve()
CONFIRM_PREFERENCES_JSON = (PROJECT_ROOT / _v2_results_rel / "confirmatory_preferences.json").resolve()
CONFIRM_MERGE_RESULTS_JSON = (PROJECT_ROOT / _v2_results_rel / "confirmatory_merge_results.json").resolve()
OUTPUT_ZIP_V2 = (PROJECT_ROOT / _v2_results_rel / "tinyllama_helpsteer2_proxy_validation_v2_outputs.zip").resolve()

BOOTSTRAP_N = 2000
BOOTSTRAP_SEED = 137
CONFIRM_NUM_PREFERENCES = 12
CONFIRM_PREF_SEED = 911
CONFIRM_PROMPT_OFFSET = 80
CONFIRM_NUM_PROMPTS = 80
RHO_M1_PENALTY = 0.5

print(f"Metrics v2 JSON: {METRICS_V2_JSON}")
print(f"Pre-registration v2 JSON: {PREREGISTRATION_V2_JSON}")
print(f"Confirmatory prompts: {CONFIRM_PROMPT_PATH}")
print(f"Confirmatory preferences: {CONFIRM_PREFERENCES_JSON}")
print(f"Confirmatory merge results: {CONFIRM_MERGE_RESULTS_JSON}")
print(f"Output zip v2: {OUTPUT_ZIP_V2}")


### Optional: clear reward cache for a clean re-run

Run this cell before reward collection when generation settings, prompts, adapters, or scoring logic changed. It removes reward-dependent outputs so the next run cannot mix old and new formats.


In [ ]:
from pathlib import Path

files_to_delete = [
    REWARD_CACHE_JSONL,
    REWARD_MATRIX_NPY,
    PROXY_RESULTS_JSON,
    PROXY_RESULTS_CSV,
    SPEARMAN_PLOT_PNG,
]

for path in files_to_delete:
    path = Path(path)
    if path.exists():
        path.unlink()
        print(f"Deleted: {path}")
    else:
        print(f"Already missing: {path}")

assert not REWARD_CACHE_JSONL.exists()
assert not REWARD_MATRIX_NPY.exists()
print("Clean reward run state confirmed.")


## 5. Optional: upload input zips

Use this for the relationship-matrix zip from notebook 05 and, if you collect rewards, the adapter zip from notebook 02.

In [ ]:
def safe_extract_zip(zip_file: zipfile.ZipFile, extract_root: Path) -> None:
    extract_root = extract_root.resolve()
    for member in zip_file.infolist():
        target_path = (extract_root / member.filename).resolve()
        if not str(target_path).startswith(str(extract_root)):
            raise ValueError(f"Unsafe zip member path: {member.filename}")
    zip_file.extractall(extract_root)


if RUN_INPUT_ZIP_UPLOAD:
    try:
        from google.colab import files
    except ImportError as error:
        raise RuntimeError("Zip upload is only available in Google Colab.") from error

    uploaded_files = files.upload()
    if not uploaded_files:
        raise RuntimeError("No zip was uploaded.")
    for uploaded_name in uploaded_files:
        uploaded_path = Path(uploaded_name)
        if uploaded_path.suffix.lower() != ".zip":
            raise ValueError(f"Expected .zip input, got: {uploaded_path}")
        print(f"Extracting {uploaded_path} to {INPUT_ZIP_EXTRACT_ROOT}")
        with zipfile.ZipFile(uploaded_path) as zip_file:
            safe_extract_zip(zip_file, INPUT_ZIP_EXTRACT_ROOT)
else:
    print("Input zip upload skipped.")

## 6. Show important files

Check these paths before continuing.

In [ ]:
important_paths = [
    CONFIG_PATH,
    COSINE_MATRIX_CSV,
    GRAM_MATRIX_CSV,
    PROJECT_ROOT / "src" / "proxy_validation.py",
    PROJECT_ROOT / "src" / "effective_lora_geometry.py",
    PROJECT_ROOT / "src" / "tinyllama_training_utils.py",
]
for path in important_paths:
    print(f"{'FOUND' if path.exists() else 'MISSING'}: {path}")

## 7. Load config, preferences, and relationship matrices

`R_cos` is the primary proxy matrix. `R_gram` is the scale-sensitive variant.

In [ ]:
from src.experiment_config import get_attribute_order, load_experiment_config, validate_preference_vectors
from src.proxy_validation import (
    analyze_geometry_only,
    build_search_set,
    collect_reward_matrix,
    load_labeled_matrix_csv,
    run_spearman_analysis,
    write_json,
    write_numpy,
)

config = load_experiment_config(CONFIG_PATH)
ATTRIBUTES = get_attribute_order(config)
PREFERENCES = validate_preference_vectors(config)
BASE_MODEL_NAME = str(config["base_model_name"])
REWARD_MODEL_NAME = str(config["reward_model_name"])
PROMPT_PATH = (PROJECT_ROOT / str(config["prompt_path"])).resolve()
ADAPTER_ROOT = (PROJECT_ROOT / str(config["adapter_dir"])).resolve()

R_cos = load_labeled_matrix_csv(COSINE_MATRIX_CSV, ATTRIBUTES)
R_gram = load_labeled_matrix_csv(GRAM_MATRIX_CSV, ATTRIBUTES)

print(f"Attributes: {ATTRIBUTES}")
print(f"Preferences: {list(PREFERENCES)}")
print(f"Base model: {BASE_MODEL_NAME}")
print(f"Reward model: {REWARD_MODEL_NAME}")
print(f"Prompt path: {PROMPT_PATH}")
print(f"Adapter root: {ADAPTER_ROOT}")
print("R_cos loaded:")
display(pd.DataFrame(R_cos, index=ATTRIBUTES, columns=ATTRIBUTES))
print("R_gram loaded:")
display(pd.DataFrame(R_gram, index=ATTRIBUTES, columns=ATTRIBUTES))

## 8. Build and freeze search set B

`B` contains vertices, the uniform point, configured preference vectors, and seeded Dirichlet samples.

In [ ]:
B = build_search_set(
    len(ATTRIBUTES),
    n_dirichlet=N_DIRICHLET,
    dirichlet_alpha=DIRICHLET_ALPHA,
    preferences=list(PREFERENCES.values()),
    seed=SEARCH_SET_SEED,
)
write_numpy(SEARCH_SET_NPY, B)

print(f"|B| = {len(B)}")
print(f"Saved B to {SEARCH_SET_NPY}")
display(pd.DataFrame(B, columns=ATTRIBUTES).head(12))

## 9. Geometry-only pre-check

Run this before spending ArmoRM compute. Tiny dynamic range means the Spearman test has low power.

In [ ]:
geometry_precheck = analyze_geometry_only(B, R_cos, R_gram, PREFERENCES)
write_json(GEOMETRY_PRECHECK_JSON, geometry_precheck)
print(json.dumps(geometry_precheck, indent=2))
print(f"Saved geometry pre-check to {GEOMETRY_PRECHECK_JSON}")

## 10. Pre-register interpretation thresholds

Run and save this before setting `RUN_REWARD_COLLECTION = True`.

In [ ]:
preregistration = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "proxy_validated_rho_threshold": PROXY_VALIDATED_RHO,
    "proxy_weak_or_negative_rho_threshold": PROXY_WEAK_RHO,
    "interpretation": {
        "rho >= proxy_validated_rho_threshold": "geometry proxy is treated as validated for this fixed B and prompt set",
        "rho <= proxy_weak_or_negative_rho_threshold": "negative or weak proxy result; valid finding, do not reinterpret after seeing results",
        "between_thresholds": "moderate evidence; report as inconclusive or partial support",
    },
    "fixed_search_set_path": str(SEARCH_SET_NPY),
    "geometry_precheck_path": str(GEOMETRY_PRECHECK_JSON),
    "reward_prompt_count": NUM_REWARD_PROMPTS,
    "heldout_prompt_split": HELDOUT_PROMPT_SPLIT,
    "proxy_prompt_path": str(PROXY_PROMPT_PATH),
    "max_new_tokens": MAX_NEW_TOKENS,
    "repetition_penalty": REPETITION_PENALTY,
    "no_repeat_ngram_size": NO_REPEAT_NGRAM_SIZE,
    "note": "ArmoRM rewards are evaluation-only and must not change training, R, N, or checkpointing.",
}
write_json(PREREGISTRATION_JSON, preregistration)
print(json.dumps(preregistration, indent=2))
print(f"Saved pre-registration to {PREREGISTRATION_JSON}")

## 11. Reward collection setup

The next cells only load TinyLlama and ArmoRM when `RUN_REWARD_COLLECTION = True`.

In [ ]:
from src.tinyllama_training_utils import ARMORM_HELPSTEER_OBJECTIVES, load_reward_prompts, model_input_device

expected_armorm_mapping = {
    "helpfulness": (0, "helpsteer-helpfulness"),
    "correctness": (1, "helpsteer-correctness"),
    "coherence": (2, "helpsteer-coherence"),
    "complexity": (3, "helpsteer-complexity"),
    "verbosity": (4, "helpsteer-verbosity"),
}
actual_armorm_mapping = {attribute: ARMORM_HELPSTEER_OBJECTIVES[attribute] for attribute in ATTRIBUTES}
if actual_armorm_mapping != expected_armorm_mapping:
    raise RuntimeError(f"Unexpected ArmoRM HelpSteer objective mapping: {actual_armorm_mapping}")

objective_indices = [ARMORM_HELPSTEER_OBJECTIVES[attribute][0] for attribute in ATTRIBUTES]
objective_names = [ARMORM_HELPSTEER_OBJECTIVES[attribute][1] for attribute in ATTRIBUTES]
print("Verified ArmoRM HelpSteer objective mapping:", dict(zip(ATTRIBUTES, objective_names)))


def write_prompt_jsonl(path: Path, prompts: list[dict[str, str]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as output_file:
        for record in prompts:
            output_file.write(json.dumps(record, ensure_ascii=False) + "\n")


def build_validation_prompt_set(num_prompts: int, seed: int) -> list[dict[str, str]]:
    from datasets import load_dataset

    dataset_name = str(config["dataset_name"])
    dataset = load_dataset(dataset_name, split=HELDOUT_PROMPT_SPLIT)
    seen = set()
    candidates = []
    for index, example in enumerate(dataset):
        prompt = str(example.get("prompt", "")).strip()
        if not prompt or prompt in seen:
            continue
        seen.add(prompt)
        candidates.append({"prompt_id": f"helpsteer2_{HELDOUT_PROMPT_SPLIT}_{index}", "prompt": prompt})
    if len(candidates) < num_prompts:
        raise RuntimeError(
            f"Only {len(candidates)} unique prompts found in {dataset_name}/{HELDOUT_PROMPT_SPLIT}, "
            f"expected {num_prompts}."
        )
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(candidates))[:num_prompts]
    return [candidates[int(index)] for index in order]


configured_prompts = load_reward_prompts(PROMPT_PATH)
if "reward_monitor" in PROMPT_PATH.name:
    raise RuntimeError(f"Refusing to use monitoring prompts for proxy validation: {PROMPT_PATH}")

if len(configured_prompts) >= NUM_REWARD_PROMPTS:
    reward_prompts = configured_prompts[:NUM_REWARD_PROMPTS]
    prompt_source = str(PROMPT_PATH)
else:
    print(
        f"Configured prompt file has only {len(configured_prompts)} prompts, "
        f"but NUM_REWARD_PROMPTS={NUM_REWARD_PROMPTS}. Building a deterministic held-out "
        f"prompt set from HelpSteer2 {HELDOUT_PROMPT_SPLIT}."
    )
    reward_prompts = build_validation_prompt_set(NUM_REWARD_PROMPTS, SEARCH_SET_SEED)
    write_prompt_jsonl(PROXY_PROMPT_PATH, reward_prompts)
    prompt_source = str(PROXY_PROMPT_PATH)

print(f"Loaded {len(reward_prompts)} fixed held-out reward prompts from {prompt_source}.")

## 12. Load TinyLlama and ArmoRM

This is the expensive model-loading cell. Skip it unless reward collection is enabled.

In [ ]:
base_model = None
generation_tokenizer = None
reward_model = None
reward_tokenizer = None

if RUN_REWARD_COLLECTION:
    import torch
    from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer

    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16 if torch.cuda.is_available() else torch.float32
    device_map = "auto" if torch.cuda.is_available() else None

    print(f"Loading TinyLlama base model with dtype={dtype}, device_map={device_map}")
    generation_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
    if generation_tokenizer.pad_token is None:
        generation_tokenizer.pad_token = generation_tokenizer.eos_token
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        torch_dtype=dtype,
        device_map=device_map,
    )
    base_model.eval()

    print(f"Loading ArmoRM reward model with dtype={dtype}, device_map={device_map}")
    reward_tokenizer = AutoTokenizer.from_pretrained(REWARD_MODEL_NAME, trust_remote_code=True)
    reward_model = AutoModelForSequenceClassification.from_pretrained(
        REWARD_MODEL_NAME,
        trust_remote_code=True,
        torch_dtype=dtype,
        device_map=device_map,
    )
    reward_model.requires_grad_(False)
    reward_model.eval()
else:
    print("RUN_REWARD_COLLECTION is False; model loading skipped.")

## 13. Build effective LoRA merge factors

This prepares exact effective LoRA deltas `scaling * (B @ A)` for temporary in-place merging.

In [ ]:
def adapter_path_for(attribute: str) -> Path:
    return ADAPTER_ROOT / f"tinyllama-helpsteer2-{attribute}-adapter"


layer_factors = None
if RUN_REWARD_COLLECTION:
    from src.effective_lora_geometry import load_effective_lora_geometry, validate_compatible_geometries

    def canonical_module_name(name: str) -> str:
        for prefix in ("base_model.model.",):
            if name.startswith(prefix):
                return name[len(prefix):]
        return name

    geometries = {attribute: load_effective_lora_geometry(adapter_path_for(attribute)) for attribute in ATTRIBUTES}
    validate_compatible_geometries([geometries[attribute] for attribute in ATTRIBUTES], ATTRIBUTES)

    layer_factors = {}
    for module_name in sorted(geometries[ATTRIBUTES[0]]):
        canonical_name = canonical_module_name(module_name)
        layer_factors[canonical_name] = [
            (
                geometries[attribute][module_name].lora_a,
                geometries[attribute][module_name].lora_b,
                geometries[attribute][module_name].scaling,
            )
            for attribute in ATTRIBUTES
        ]

    model_modules = dict(base_model.named_modules())
    missing_modules = [name for name in layer_factors if name not in model_modules]
    if missing_modules:
        raise RuntimeError(f"LoRA modules do not match TinyLlama module names. First missing modules: {missing_modules[:5]}")
    print(f"Prepared effective LoRA factors for {len(layer_factors)} modules.")
else:
    print("RUN_REWARD_COLLECTION is False; LoRA merge-factor setup skipped.")

## 14. Define merge, generation, and ArmoRM scoring

The merge context adds effective LoRA deltas before generation and subtracts them again afterward.

In [ ]:
if RUN_REWARD_COLLECTION:
    import torch

    def compute_weight_delta(weight, factors, lam: np.ndarray):
        compute_dtype = torch.float32 if weight.dtype in {torch.float16, torch.bfloat16} else weight.dtype
        delta = torch.zeros(weight.shape, device=weight.device, dtype=compute_dtype)
        for index, (lora_a, lora_b, scaling) in enumerate(factors):
            coefficient = float(lam[index])
            if coefficient == 0.0:
                continue
            A = lora_a.to(device=weight.device, dtype=compute_dtype)
            B_factor = lora_b.to(device=weight.device, dtype=compute_dtype)
            delta.add_(coefficient * float(scaling) * (B_factor @ A))
        return delta.to(dtype=weight.dtype)

    @contextmanager
    def merged_effective_lora_weights(model, factors_by_module: dict, lam: np.ndarray):
        modules = dict(model.named_modules())
        originals = {}
        try:
            for module_name, factors in factors_by_module.items():
                module = modules[module_name]
                weight = module.weight
                originals[module_name] = weight.detach().clone()
                delta = compute_weight_delta(weight, factors, lam)
                weight.data.add_(delta)
                del delta
            yield model
        finally:
            for module_name, original_weight in originals.items():
                modules[module_name].weight.data.copy_(original_weight)
            del originals
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    def generate_answer(prompt: str) -> str:
        device = model_input_device(base_model)
        # Match the SFT training format from format_helpsteer2_text exactly.
        # ArmoRM scoring below still uses the reward model's own chat template.
        text = f"Human: {prompt}\n\nAssistant: "
        encoded = generation_tokenizer(text, return_tensors="pt")
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        with torch.inference_mode():
            generated = base_model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                num_beams=1,
                repetition_penalty=REPETITION_PENALTY,
                no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
                pad_token_id=generation_tokenizer.eos_token_id,
            )
        return generation_tokenizer.decode(generated[0, input_ids.shape[1]:], skip_special_tokens=True).strip()

    def score_answer_heads(prompt: str, answer: str) -> np.ndarray:
        messages = [{"role": "user", "content": prompt}, {"role": "assistant", "content": answer}]
        if getattr(reward_tokenizer, "chat_template", None):
            inputs = {"input_ids": reward_tokenizer.apply_chat_template(messages, return_tensors="pt")}
        else:
            text = f"Human: {prompt}\n\nAssistant: {answer}"
            inputs = reward_tokenizer(text, return_tensors="pt", truncation=True)
        reward_device = next(reward_model.parameters()).device
        inputs = {key: value.to(reward_device) for key, value in inputs.items()}
        with torch.inference_mode():
            outputs = reward_model(**inputs)
        rewards = getattr(outputs, "rewards", None)
        if rewards is None:
            raise RuntimeError("ArmoRM output has no .rewards tensor.")
        tensor = torch.as_tensor(rewards).detach().float()
        if tensor.ndim == 1:
            tensor = tensor.unsqueeze(0)
        return tensor[0, objective_indices].cpu().numpy().astype(np.float64)

    def reward_of_lambda(lam: np.ndarray) -> np.ndarray:
        per_prompt = []
        started = time.perf_counter()
        with merged_effective_lora_weights(base_model, layer_factors, lam):
            base_model.eval()
            for record in reward_prompts:
                answer = generate_answer(record["prompt"])
                per_prompt.append(score_answer_heads(record["prompt"], answer))
        reward = np.mean(np.asarray(per_prompt, dtype=np.float64), axis=0)
        print(f"lambda={np.round(lam, 3)} mean_reward={np.round(reward, 4)} time={time.perf_counter() - started:.1f}s")
        return reward
else:
    print("RUN_REWARD_COLLECTION is False; reward function setup skipped.")

## 14a. Qualitative endpoint comparison: coherence vs. verbosity

Generate answers for the coherence and verbosity simplex vertices on the same prompt. This sanity check makes the qualitative difference in length and style visible before collecting the full reward matrix.

In [ ]:
DEMO_ENDPOINT_PROMPT_INDEX = 0
DEMO_ENDPOINT_ATTRIBUTES = ["coherence", "verbosity"]

if RUN_REWARD_COLLECTION:
    from IPython.display import Markdown, display

    if not reward_prompts:
        raise RuntimeError("No reward prompts are loaded for endpoint comparison.")
    if DEMO_ENDPOINT_PROMPT_INDEX >= len(reward_prompts):
        raise IndexError(
            f"DEMO_ENDPOINT_PROMPT_INDEX={DEMO_ENDPOINT_PROMPT_INDEX} but only "
            f"{len(reward_prompts)} reward prompts are available."
        )

    demo_prompt = reward_prompts[DEMO_ENDPOINT_PROMPT_INDEX]["prompt"]

    def endpoint_lambda(attribute: str) -> np.ndarray:
        lam = np.zeros(len(ATTRIBUTES), dtype=np.float64)
        lam[ATTRIBUTES.index(attribute)] = 1.0
        return lam

    display(Markdown("### Shared prompt"))
    display(Markdown(f"```text\n{demo_prompt}\n```"))

    endpoint_rows = []
    for attribute in DEMO_ENDPOINT_ATTRIBUTES:
        lam = endpoint_lambda(attribute)
        with merged_effective_lora_weights(base_model, layer_factors, lam):
            base_model.eval()
            answer = generate_answer(demo_prompt)

        scores = score_answer_heads(demo_prompt, answer)
        token_count = len(generation_tokenizer(answer, add_special_tokens=False)["input_ids"])
        score_text = ", ".join(
            f"{name}={value:.4f}" for name, value in zip(ATTRIBUTES, scores)
        )

        endpoint_rows.append(
            {
                "endpoint": attribute,
                "answer_chars": len(answer),
                "answer_tokens": token_count,
                **{f"armorm_{name}": value for name, value in zip(ATTRIBUTES, scores)},
            }
        )

        display(Markdown(f"### {attribute} endpoint"))
        display(Markdown(f"**Length:** {len(answer)} chars / {token_count} tokens  "))
        display(Markdown(f"**ArmoRM scores:** {score_text}"))
        display(Markdown(f"```text\n{answer}\n```"))

    endpoint_demo_df = pd.DataFrame(endpoint_rows)
    display(endpoint_demo_df)
else:
    print("RUN_REWARD_COLLECTION is False; endpoint comparison skipped.")


## 15. Collect or load reward matrix

Reward collection is cached in JSONL and saved as `.npy` when complete.

In [ ]:
Reward = None
if RUN_REWARD_COLLECTION:
    Reward = collect_reward_matrix(B, reward_of_lambda, REWARD_CACHE_JSONL)
    write_numpy(REWARD_MATRIX_NPY, Reward)
    print(f"Saved reward matrix to {REWARD_MATRIX_NPY}")
elif REWARD_MATRIX_NPY.is_file():
    Reward = np.load(REWARD_MATRIX_NPY)
    print(f"Loaded reward matrix from {REWARD_MATRIX_NPY}")
else:
    print(
        "Reward matrix not found and RUN_REWARD_COLLECTION is False. "
        "Geometry-only and pre-registration outputs are complete. "
        "Set RUN_REWARD_COLLECTION = True, or upload reward_matrix.npy, to run Spearman analysis."
    )

if Reward is not None:
    reward_df = pd.DataFrame(Reward, columns=ATTRIBUTES)
    display(reward_df.head())

## 16. Run Spearman proxy analysis

This compares geometry-score ranks against preference-weighted ArmoRM utility ranks over the fixed search set `B`.

In [ ]:
proxy_results = None
if Reward is None:
    print("Skipping Spearman analysis because Reward is not available yet.")
else:
    proxy_results = run_spearman_analysis(B, Reward, R_cos, R_gram, PREFERENCES)
    write_json(PROXY_RESULTS_JSON, proxy_results)
    print(json.dumps(proxy_results["aggregate"], indent=2))
    print(f"Saved proxy results to {PROXY_RESULTS_JSON}")

## 16a. Metric set v2 (rank-U_p, ?m%, bootstrap CI)

This block uses the already collected reward matrix only. It does not call TinyLlama or ArmoRM.


In [ ]:
from scipy.stats import rankdata, spearmanr

metrics_v2 = None
metrics_v2_df = pd.DataFrame()


def rank_normalize(reward: np.ndarray) -> np.ndarray:
    reward = np.asarray(reward, dtype=np.float64)
    if reward.ndim != 2:
        raise ValueError("reward must be a two-dimensional matrix.")
    n = reward.shape[0]
    if n <= 1:
        return np.zeros_like(reward, dtype=np.float64)
    normalized = np.zeros_like(reward, dtype=np.float64)
    for column in range(reward.shape[1]):
        normalized[:, column] = (rankdata(reward[:, column], method="average") - 1.0) / (n - 1.0)
    return normalized


def u_p(reward_norm: np.ndarray, p: np.ndarray) -> np.ndarray:
    return np.asarray(reward_norm, dtype=np.float64) @ np.asarray(p, dtype=np.float64)


def delta_m_percent(reward_row: np.ndarray, p: np.ndarray) -> float:
    stl = np.array([Reward[k, k] for k in range(len(ATTRIBUTES))], dtype=np.float64)
    reward_row = np.asarray(reward_row, dtype=np.float64)
    p = np.asarray(p, dtype=np.float64)
    return float(np.sum(p * (reward_row - stl) / stl) * 100.0)


def _spearman_or_nan(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    if x.size < 2 or y.size < 2:
        return float("nan")
    if np.allclose(x, x[0]) or np.allclose(y, y[0]):
        return float("nan")
    rho, _ = spearmanr(x, y)
    return float(rho)


def bootstrap_ci(x: np.ndarray, y: np.ndarray, n: int = BOOTSTRAP_N, seed: int = BOOTSTRAP_SEED) -> tuple[float, float, float]:
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    if x.shape != y.shape:
        raise ValueError("x and y must have the same shape.")
    rho = _spearman_or_nan(x, y)
    rng = np.random.default_rng(seed)
    boot = []
    for _ in range(int(n)):
        indices = rng.integers(0, x.size, size=x.size)
        value = _spearman_or_nan(x[indices], y[indices])
        if np.isfinite(value):
            boot.append(value)
    if not boot:
        return rho, float("nan"), float("nan")
    lo, hi = np.percentile(np.asarray(boot, dtype=np.float64), [2.5, 97.5])
    return rho, float(lo), float(hi)


def selection_regret(score: np.ndarray, u: np.ndarray) -> float:
    score = np.asarray(score, dtype=np.float64)
    u = np.asarray(u, dtype=np.float64)
    return float(u.max() - u[int(np.argmax(score))])


def random_regret(u: np.ndarray, n: int = 400, seed: int = BOOTSTRAP_SEED) -> float:
    u = np.asarray(u, dtype=np.float64)
    rng = np.random.default_rng(seed)
    choices = rng.integers(0, u.size, size=int(n))
    return float(np.mean(u.max() - u[choices]))


def closest_row_index(matrix: np.ndarray, vector: np.ndarray, atol: float = 1e-8) -> int:
    distances = np.linalg.norm(np.asarray(matrix, dtype=np.float64) - np.asarray(vector, dtype=np.float64), axis=1)
    index = int(np.argmin(distances))
    if distances[index] > atol:
        print(f"Warning: no exact B row found for p; using closest row {index} with distance {distances[index]:.3e}.")
    return index


if Reward is None:
    print("Skipping metric set v2 because Reward is not available yet.")
else:
    reward_rank = rank_normalize(Reward)
    vertex_matrix = reward_rank[: len(ATTRIBUTES)].T
    rows = []
    for preference_name, preference_values in PREFERENCES.items():
        p = np.asarray(preference_values, dtype=np.float64)
        rank_utility = u_p(reward_rank, p)
        score_r = B @ (R_cos @ p)
        score_h = B @ (vertex_matrix @ p)
        rho_r, rho_r_lo, rho_r_hi = bootstrap_ci(score_r, rank_utility)
        rho_h = _spearman_or_nan(score_h, rank_utility)
        p_index = closest_row_index(B, p)
        quality_mass = float(p[:3].sum())
        complexity_verbosity_mass = float(p[3:].sum())
        axis = "complexity_verbosity" if complexity_verbosity_mass > quality_mass else "quality"
        rho_r_ci_excludes_zero = bool(np.isfinite(rho_r_lo) and np.isfinite(rho_r_hi) and (rho_r_lo > 0.0 or rho_r_hi < 0.0))
        rows.append(
            {
                "preference": preference_name,
                "axis": axis,
                "complexity_verbosity_mass": complexity_verbosity_mass,
                "quality_mass": quality_mass,
                "rank_U_p_at_p": float(rank_utility[p_index]),
                "delta_m_pct": delta_m_percent(Reward[p_index], p),
                "rho_R": rho_r,
                "rho_R_ci_lo": rho_r_lo,
                "rho_R_ci_hi": rho_r_hi,
                "rho_R_ci_excludes_zero": rho_r_ci_excludes_zero,
                "rho_H": rho_h,
                "regret_R": selection_regret(score_r, rank_utility),
                "regret_random": random_regret(rank_utility, seed=BOOTSTRAP_SEED),
            }
        )
    metrics_v2_df = pd.DataFrame(rows)
    metrics_v2 = {
        "metric_set": "v2_rank_utility_delta_m_bootstrap_regret",
        "attributes": ATTRIBUTES,
        "reward_shape": list(Reward.shape),
        "bootstrap_n": BOOTSTRAP_N,
        "bootstrap_seed": BOOTSTRAP_SEED,
        "stl_reference": {attribute: float(Reward[index, index]) for index, attribute in enumerate(ATTRIBUTES)},
        "rows": rows,
    }
    write_json(METRICS_V2_JSON, metrics_v2)
    print(f"Saved metric set v2 to {METRICS_V2_JSON}")
    display(metrics_v2_df)


## 16b. Fresh confirmatory prompts

These prompts are selected from the same held-out split, using a disjoint slice after the first reward-prompt block.


In [ ]:
def build_validation_prompt_slice(offset: int, num_prompts: int, seed: int) -> list[dict[str, str]]:
    from datasets import load_dataset

    dataset_name = str(config["dataset_name"])
    dataset = load_dataset(dataset_name, split=HELDOUT_PROMPT_SPLIT)
    seen = set()
    candidates = []
    for index, example in enumerate(dataset):
        prompt = str(example.get("prompt", "")).strip()
        if not prompt or prompt in seen:
            continue
        seen.add(prompt)
        candidates.append({"prompt_id": f"helpsteer2_{HELDOUT_PROMPT_SPLIT}_{index}", "prompt": prompt})
    required = int(offset) + int(num_prompts)
    if len(candidates) < required:
        raise RuntimeError(
            f"Only {len(candidates)} unique prompts found in {dataset_name}/{HELDOUT_PROMPT_SPLIT}; "
            f"need at least {required} for offset={offset} and num_prompts={num_prompts}."
        )
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(candidates))
    selected = [candidates[int(index)] for index in order[int(offset):required]]
    current_prompts = {record["prompt"] for record in reward_prompts}
    overlap = current_prompts.intersection({record["prompt"] for record in selected})
    if overlap:
        raise RuntimeError(
            f"Confirmatory prompt slice overlaps with the current reward_prompts ({len(overlap)} prompts). "
            "Increase CONFIRM_PROMPT_OFFSET or verify the prompt-selection seed."
        )
    return selected


confirm_reward_prompts = build_validation_prompt_slice(
    CONFIRM_PROMPT_OFFSET,
    CONFIRM_NUM_PROMPTS,
    SEARCH_SET_SEED,
)
write_prompt_jsonl(CONFIRM_PROMPT_PATH, confirm_reward_prompts)
print(f"Saved {len(confirm_reward_prompts)} confirmatory prompts to {CONFIRM_PROMPT_PATH}")
display(pd.DataFrame(confirm_reward_prompts).head())


## 16c. Confirmatory complexity/verbosity preferences

These preferences are frozen before the confirmatory ArmoRM merge evaluation.


In [ ]:
rng = np.random.default_rng(CONFIRM_PREF_SEED)
confirm_alpha = np.array([0.3, 0.3, 0.3, 2.5, 2.5], dtype=np.float64)
confirm_preferences = []
for index, vector in enumerate(rng.dirichlet(confirm_alpha, size=CONFIRM_NUM_PREFERENCES)):
    confirm_preferences.append(
        {
            "name": f"conf_{index:02d}",
            "preference": [float(value) for value in vector],
            "alpha": confirm_alpha.tolist(),
            "seed": CONFIRM_PREF_SEED,
        }
    )
write_json(CONFIRM_PREFERENCES_JSON, confirm_preferences)
print(f"Saved {len(confirm_preferences)} confirmatory preferences to {CONFIRM_PREFERENCES_JSON}")
display(pd.DataFrame([{ "name": row["name"], **{attribute: row["preference"][i] for i, attribute in enumerate(ATTRIBUTES)} } for row in confirm_preferences]))


## 16d. Pre-registration v2

This freezes the additional metric and confirmatory protocol before any new confirmatory ArmoRM calls.


In [ ]:
preregistration_v2 = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "confirmatory": True,
    "metrics": {
        "primary": "rank-normalized U_p computed from per-head reward ranks over B",
        "secondary": "delta_m_percent using STL=Reward[k,k] from simplex vertices",
        "bootstrap_ci": "95% Spearman CI from bootstrap resampling over B indices; effect is treated as real only if CI excludes 0",
        "selection_regret": "u.max() - u[argmax(score)] compared with random selection regret per axis",
        "rho_m1_penalty": RHO_M1_PENALTY,
    },
    "thresholds": {
        "rho_validated": 0.60,
        "rho_weak_or_negative": 0.30,
        "effect_real": "Bootstrap 95% CI excludes 0",
    },
    "confirm_prompt_path": str(CONFIRM_PROMPT_PATH),
    "confirm_preferences_json": str(CONFIRM_PREFERENCES_JSON),
    "confirm_num_prompts": CONFIRM_NUM_PROMPTS,
    "confirm_prompt_offset": CONFIRM_PROMPT_OFFSET,
    "confirm_num_preferences": CONFIRM_NUM_PREFERENCES,
    "confirm_pref_seed": CONFIRM_PREF_SEED,
    "heldout_prompt_split": HELDOUT_PROMPT_SPLIT,
    "reward_prompt_count_original": NUM_REWARD_PROMPTS,
    "note": "metrics frozen before confirmatory ArmoRM contact; ArmoRM evaluation-only",
}
write_json(PREREGISTRATION_V2_JSON, preregistration_v2)
print(json.dumps(preregistration_v2, indent=2))
print(f"Saved pre-registration v2 to {PREREGISTRATION_V2_JSON}")


## 17. Save summary CSV

The CSV is convenient for plotting and thesis tables.

In [ ]:
summary_df = pd.DataFrame()
if proxy_results is None:
    print("Skipping summary CSV because proxy_results is not available yet.")
else:
    summary_rows = []
    for preference_name, entry in proxy_results["per_preference"].items():
        for score_name, rho in entry["rho"].items():
            matrix_name, geometry_score = score_name.split("_", 1)
            summary_rows.append(
                {
                    "preference_name": preference_name,
                    "matrix": matrix_name,
                    "geometry_score": geometry_score,
                    "spearman_rho": rho,
                    "U_best": entry["U_best"],
                    "U_at_p_baseline": entry.get("U_at_p_baseline"),
                    "gap_best_vs_p": entry.get("gap_best_vs_p"),
                }
            )
    summary_df = pd.DataFrame(summary_rows)
    PROXY_RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
    summary_df.to_csv(PROXY_RESULTS_CSV, index=False)
    print(f"Saved summary CSV to {PROXY_RESULTS_CSV}")
    display(summary_df)

## 18. Plot Spearman summary

The dashed lines are the pre-registered interpretation thresholds.

In [ ]:
if summary_df.empty:
    print("Skipping Spearman plot because summary_df is empty.")
else:
    plot_df = summary_df.dropna(subset=["spearman_rho"]).copy()
    plot_df["label"] = plot_df["preference_name"] + " / " + plot_df["matrix"] + " / " + plot_df["geometry_score"]
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(12, 6), dpi=160)
    ax.bar(plot_df["label"], plot_df["spearman_rho"], color="#4C78A8")
    ax.axhline(PROXY_VALIDATED_RHO, color="#2CA02C", linestyle="--", linewidth=1.2, label="validated threshold")
    ax.axhline(PROXY_WEAK_RHO, color="#D62728", linestyle="--", linewidth=1.2, label="weak threshold")
    ax.set_ylim(-1.0, 1.0)
    ax.set_ylabel("Spearman rho")
    ax.set_title("TinyLlama HelpSteer2 Proxy Validation")
    ax.tick_params(axis="x", labelrotation=75)
    ax.legend()
    fig.tight_layout()
    fig.savefig(SPEARMAN_PLOT_PNG, bbox_inches="tight")
    plt.show()
    print(f"Saved plot to {SPEARMAN_PLOT_PNG}")

## 19. Validate outputs

These checks ensure that the experiment produced finite matrices and saved the expected files.

In [ ]:
assert B.ndim == 2 and B.shape[1] == len(ATTRIBUTES), "B must be |B| x m."
assert SEARCH_SET_NPY.is_file(), f"Missing {SEARCH_SET_NPY}"
assert GEOMETRY_PRECHECK_JSON.is_file(), f"Missing {GEOMETRY_PRECHECK_JSON}"
assert PREREGISTRATION_JSON.is_file(), f"Missing {PREREGISTRATION_JSON}"
if Reward is None:
    print("Geometry-only validation passed. Reward/Spearman validation skipped because Reward is not available yet.")
else:
    assert Reward.shape == B.shape, "Reward matrix must have the same shape as B."
    assert np.isfinite(Reward).all(), "Reward matrix contains non-finite values."
    assert PROXY_RESULTS_JSON.is_file(), f"Missing {PROXY_RESULTS_JSON}"
    assert PROXY_RESULTS_CSV.is_file(), f"Missing {PROXY_RESULTS_CSV}"
    assert SPEARMAN_PLOT_PNG.is_file(), f"Missing {SPEARMAN_PLOT_PNG}"
    print("Full proxy-validation output validation passed.")

## 19a. Confirmatory M1+ merge evaluation on fresh prompts

This evaluates ?* and ?=p on the fresh confirmatory prompt set using the existing merge, generation, and ArmoRM scoring path.


In [ ]:
confirm_merge_df = pd.DataFrame()
confirm_merge_results = None

if not RUN_REWARD_COLLECTION:
    print("Skipping confirmatory merge evaluation because RUN_REWARD_COLLECTION is False.")
elif Reward is None:
    print("Skipping confirmatory merge evaluation because Reward is not available yet.")
else:
    from scipy.optimize import minimize

    def load_prompt_jsonl(path: Path) -> list[dict[str, str]]:
        records = []
        for line in Path(path).read_text(encoding="utf-8").splitlines():
            if line.strip():
                records.append(json.loads(line))
        return records

    def load_confirm_preferences(path: Path) -> list[dict]:
        payload = json.loads(Path(path).read_text(encoding="utf-8"))
        if not isinstance(payload, list):
            raise ValueError("Confirmatory preferences JSON must contain a list.")
        return payload

    def lambda_m1plus(preference: np.ndarray, penalty: float = RHO_M1_PENALTY) -> tuple[np.ndarray, bool, str]:
        p = np.asarray(preference, dtype=np.float64)
        m = p.size

        def objective(lam: np.ndarray) -> float:
            lam = np.asarray(lam, dtype=np.float64)
            geometry_score = float(p @ (R_cos @ lam))
            trust_penalty = float((lam - p) @ R_cos @ (lam - p))
            return -(geometry_score - float(penalty) * trust_penalty)

        constraints = ({"type": "eq", "fun": lambda lam: float(np.sum(lam) - 1.0)},)
        bounds = [(0.0, 1.0)] * m
        rng = np.random.default_rng(CONFIRM_PREF_SEED)
        starts = [p, np.full(m, 1.0 / m), *np.eye(m), *rng.dirichlet(np.ones(m), size=8)]
        best = None
        messages = []
        for start in starts:
            result = minimize(objective, start, method="SLSQP", bounds=bounds, constraints=constraints, options={"maxiter": 500, "ftol": 1e-12})
            messages.append(str(result.message))
            if result.success and np.all(np.isfinite(result.x)):
                candidate = np.clip(result.x, 0.0, 1.0)
                candidate = candidate / candidate.sum()
                value = objective(candidate)
                if best is None or value < best[0]:
                    best = (value, candidate, str(result.message))
        if best is None:
            return p.copy(), False, "; ".join(messages[:3])
        return best[1], True, best[2]

    confirm_reward_prompts_loaded = load_prompt_jsonl(CONFIRM_PROMPT_PATH)
    if len(confirm_reward_prompts_loaded) != CONFIRM_NUM_PROMPTS:
        raise RuntimeError(f"Expected {CONFIRM_NUM_PROMPTS} confirmatory prompts, got {len(confirm_reward_prompts_loaded)}.")
    confirm_preferences_loaded = load_confirm_preferences(CONFIRM_PREFERENCES_JSON)

    original_reward_prompts = reward_prompts
    confirm_rows = []
    try:
        reward_prompts = confirm_reward_prompts_loaded
        for item in confirm_preferences_loaded:
            name = str(item["name"])
            p = np.asarray(item["preference"], dtype=np.float64)
            lam_star, solver_success, solver_message = lambda_m1plus(p)
            reward_lam_star = reward_of_lambda(lam_star)
            reward_p = reward_of_lambda(p)
            u_p_baseline = float(p @ reward_p)
            u_p_lam_star = float(p @ reward_lam_star)
            delta_m_p = delta_m_percent(reward_p, p)
            delta_m_lam_star = delta_m_percent(reward_lam_star, p)
            confirm_rows.append(
                {
                    "preference": name,
                    "solver_success": bool(solver_success),
                    "solver_message": solver_message,
                    "preference_vector": p.tolist(),
                    "lambda_m1plus": lam_star.tolist(),
                    "lambda_distance_l2": float(np.linalg.norm(lam_star - p)),
                    "U_p_p": u_p_baseline,
                    "U_p_lambda_star": u_p_lam_star,
                    "delta_U_p": u_p_lam_star - u_p_baseline,
                    "delta_m_pct_p": delta_m_p,
                    "delta_m_pct_lambda_star": delta_m_lam_star,
                    "delta_m_pct_gain": delta_m_lam_star - delta_m_p,
                    "reward_p": reward_p.tolist(),
                    "reward_lambda_star": reward_lam_star.tolist(),
                }
            )
    finally:
        reward_prompts = original_reward_prompts

    confirm_merge_results = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "confirm_prompt_path": str(CONFIRM_PROMPT_PATH),
        "confirm_preferences_json": str(CONFIRM_PREFERENCES_JSON),
        "rho_m1_penalty": RHO_M1_PENALTY,
        "rows": confirm_rows,
        "note": "Generated with existing reward_of_lambda path; ArmoRM evaluation-only.",
    }
    write_json(CONFIRM_MERGE_RESULTS_JSON, confirm_merge_results)
    confirm_merge_df = pd.DataFrame(
        [
            {
                "preference": row["preference"],
                "||lambda*-p||": row["lambda_distance_l2"],
                "U_p(p)": row["U_p_p"],
                "U_p(lambda*)": row["U_p_lambda_star"],
                "delta_U_p": row["delta_U_p"],
                "delta_m%(p)": row["delta_m_pct_p"],
                "delta_m%(lambda*)": row["delta_m_pct_lambda_star"],
                "delta_delta_m%": row["delta_m_pct_gain"],
            }
            for row in confirm_rows
        ]
    )
    print(f"Saved confirmatory merge results to {CONFIRM_MERGE_RESULTS_JSON}")
    display(confirm_merge_df)


## 20. Create and download output zip

This zip contains the proxy-validation outputs. Keep it out of GitHub.

In [ ]:
files_to_zip = [
    SEARCH_SET_NPY,
    GEOMETRY_PRECHECK_JSON,
    PREREGISTRATION_JSON,
    REWARD_CACHE_JSONL,
    REWARD_MATRIX_NPY,
    PROXY_RESULTS_JSON,
    PROXY_RESULTS_CSV,
    SPEARMAN_PLOT_PNG,
]
with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zip_file:
    for file_path in files_to_zip:
        if file_path.is_file():
            zip_file.write(file_path, arcname=file_path.relative_to(PROJECT_ROOT))
print(f"Created {OUTPUT_ZIP}")

try:
    from google.colab import files
    files.download(str(OUTPUT_ZIP))
except ImportError:
    print(f"Download manually from: {OUTPUT_ZIP}")

## 20a. Create and download output zip v2

This zip extends the original proxy-validation output with the v2 metrics and confirmatory files. Keep it out of GitHub.


In [ ]:
files_to_zip_v2 = [
    *files_to_zip,
    METRICS_V2_JSON,
    PREREGISTRATION_V2_JSON,
    CONFIRM_PROMPT_PATH,
    CONFIRM_PREFERENCES_JSON,
    CONFIRM_MERGE_RESULTS_JSON,
]
with zipfile.ZipFile(OUTPUT_ZIP_V2, "w", compression=zipfile.ZIP_DEFLATED) as zip_file:
    for file_path in files_to_zip_v2:
        if file_path.is_file():
            zip_file.write(file_path, arcname=file_path.relative_to(PROJECT_ROOT))
print(f"Created {OUTPUT_ZIP_V2}")

try:
    from google.colab import files
    files.download(str(OUTPUT_ZIP_V2))
except ImportError:
    print(f"Download manually from: {OUTPUT_ZIP_V2}")


## 21. Git safety check

Generated proxy-validation outputs are analysis artifacts and should normally stay out of commits.

In [ ]:
paths_to_check = [RESULTS_DIR, OUTPUT_ZIP]
print("Git ignore status:")
for path in paths_to_check:
    relative_path = path.relative_to(PROJECT_ROOT)
    result = subprocess.run(["git", "check-ignore", str(relative_path)], cwd=PROJECT_ROOT, text=True, capture_output=True, check=False)
    print(f"  {'ignored' if result.returncode == 0 else 'NOT ignored'}: {relative_path}")

subprocess.run(["git", "status", "--short", "--", *[str(path.relative_to(PROJECT_ROOT)) for path in paths_to_check]], cwd=PROJECT_ROOT, check=False)

## 22. Next step

Use the proxy-validation result to decide how strongly `R_cos` should be trusted as a relationship proxy.

If `R_cos` has weak or negative Spearman correlation with `U_p`, report that as a valid negative result and do not reinterpret the geometry proxy after seeing the reward results.